### Complete end to end pipeline for ai authorizatioin and stylometric detection

In [1]:
import pandas as pd
import numpy as np

In [3]:
from google.colab import files

uploaded = files.upload()

Saving authors_7.csv to authors_7.csv


In [4]:
df = pd.read_csv('./authors_7.csv')
df.head()

,author,content
0,H.P. Lovecraft,THE ADVENTURES OF TOM SAWYER By Mark Twain (Sa...
1,H.P. Lovecraft,Mode of Egress Tom’s Effort at Prayer Muff Pot...
2,H.P. Lovecraft,No answer. The old lady pulled her spectacles ...
3,H.P. Lovecraft,"the heart to lash him, somehow. Every time I l..."
4,H.P. Lovecraft,saying is—better’n you look. _This_ time.” She...


Now removing html tags using regular expressions :::::

In [5]:
import re

In [6]:
def remove_html_tags(text):
    clean = re.sub(r'<.*?>', '', text)
    return clean

Data may be taken from multiple places and it may contains html tags

In [7]:
df['content'].apply(remove_html_tags)

,content
0,THE ADVENTURES OF TOM SAWYER By Mark Twain (Sa...
1,Mode of Egress Tom’s Effort at Prayer Muff Pot...
2,No answer. The old lady pulled her spectacles ...
3,"the heart to lash him, somehow. Every time I l..."
4,saying is—better’n you look. _This_ time.” She...
...,...
1456,time and met Mr. Carter outside. After arrangi...
1457,crime.... Strange that men of brains had never...
1458,be a fake. No girl could deceive ME!... “...Th...
1459,about a special license to-morrow morning.” “O...


Now lets make some basic preprocessing::::>>>

In [8]:
import nltk

In [9]:
from nltk.tokenize import word_tokenize , sent_tokenize
from nltk import pos_tag
from collections import Counter
import string

Stylometric frature extraction ::::

In [10]:
def extract_feature(text):
    words = word_tokenize(text)
    sentences = sent_tokenize(text)
    if(len(words)==0): return {}
    tags = pos_tag(words)
    pos_counts = Counter(tag for _, tag in tags)
    features = {
        "char_count": len(text),
        "word_count": len(words),
        "sent_count": len(sentences),
        "avg_word_len" : sum(len(w) for w in words)/len(words),
        "avg_sentence_len" : sum(len(sen) for sen in sentences)/len(sentences),
        "unique_word_count" : len(set(words)),
        "type_token_ratio" : len(set(words)) / len(words),
        "noun_ratio" : pos_counts['NN']/len(tags),
        "verb_ratio": pos_counts["VB"] / len(tags),
        "adj_ratio": pos_counts["JJ"] / len(tags),
        "adv_ratio": pos_counts["RB"] / len(tags),
        'punctuation_count' : sum(1 for char in text if char in string.punctuation),
        'upper_case_count' : sum(1 for word in words if word.isupper()),
        }

    for tag, count in pos_counts.items():
        features[f'pos_{tag}'] = count
        ###-------------------------------------------------------------------------------------

    from nltk.corpus import stopwords
    nltk.download('stopwords')
    stop_words = set(stopwords.words('english'))
    features['stopword_count'] = sum(1 for word in words if word.lower() in stop_words)
    features['stopword_ratio'] = sum(1 for word in words if word.lower() in stop_words) / len(words) if len(words) > 0 else 0
    return features

In [13]:
import nltk

In [16]:
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [17]:
stylometric_feat = df['content'].apply(extract_feature)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_

In [18]:
stylometric_df = stylometric_feat.apply(pd.Series)

In [19]:
stylometric_df = stylometric_df.fillna(0)

Vectorization of Content: Conversion of text to vector. For making the better model
training and Stylometric detection

### Here we use TfidfVectorizer

In [20]:
### for conversion of text content to number so that we can apply model on it also
## and can make stylometric detection
from sklearn.feature_extraction.text import TfidfVectorizer

In [21]:
# Creating Instance of TfidfVectorizer
vectorizer = vectorizer = TfidfVectorizer(
    ngram_range=(1 , 2),
    max_features=10000,
    min_df=2 ,
    max_df=0.9
    )

In [22]:
# Converting df[content] into vector
X_ngram = vectorizer.fit_transform(df['content'].astype(str))

In [23]:
# sparse matrix of dtype 'float64'
X_ngram

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 508779 stored elements and shape (1461, 10000)>

Here we have X_ngram as sparse matrix.
and
stylometric_df as df with multiple features


Now We first requried to scale stylometric_df with standard scaler

In [24]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [25]:
scaled_stylometric_df = scaler.fit_transform(stylometric_df)

Now We have final scaled_stylometric_df ready to operate.

But we have to convert it also into matrix to operate

In [26]:
from scipy.sparse import csr_matrix
X_style_sparse = csr_matrix(scaled_stylometric_df)

In [27]:
# Now combine it to final combined matrix ::::
from scipy.sparse import hstack
final_df = hstack([X_ngram , X_style_sparse])

Here final_df contains the combined matrix

Now lets make the Train-Test split : 80% and 20%

In [28]:
from sklearn.model_selection import train_test_split

X_train , X_test , y_train , y_test = train_test_split(
    final_df,
    df['author'],
    test_size=0.2,
    random_state=42
)

Making Label Encoding

In [29]:
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train)
y_test = label_encoder.transform(y_test)

In [30]:
## Now lets apply the model
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
    )

In [31]:
model.fit(X_train , y_train)

RandomForestClassifier(n_estimators=200, random_state=42)

In [32]:
# %%
# from sklearn.linear_model import LogisticRegression
# model = LogisticRegression
# (
# max_iter=1000,
# random_state=42
# )
# model.fit(X_train , y_train)

In [33]:
y_pred = model.predict(X_test)

In [34]:
from sklearn.metrics import accuracy_score, classification_report
print("Accuracy :", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy : 0.9658703071672355
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        25
           1       0.96      1.00      0.98        48
           2       1.00      0.96      0.98        27
           3       0.95      1.00      0.98        83
           4       1.00      0.92      0.96        39
           5       0.94      1.00      0.97        64
           6       1.00      0.14      0.25         7

    accuracy                           0.97       293
   macro avg       0.98      0.86      0.87       293
weighted avg       0.97      0.97      0.96       293



Thanking You :::>>>

Now lets save the model

In [35]:
import joblib
joblib.dump(model, "model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(label_encoder, "label_encoder.pkl")
joblib.dump(vectorizer , "vectorizer.pkl")

['vectorizer.pkl']